In [18]:
import numpy as np
from sklearn import svm
from sklearn.metrics import accuracy_score, f1_score

train_sentences = np.load('data/training_sentences.npy', allow_pickle=True)
test_sentences = np.load('data/test_sentences.npy', allow_pickle=True)
train_labels = np.load('data/training_labels.npy', allow_pickle=True)
test_labels = np.load('data/test_labels.npy', allow_pickle=True)


def normalize_data(train_data, test_data, type=None):
    if type is None:
        return train_data, test_data

    if type == 'standard':
        mean = np.mean(train_data, axis=0)
        std = np.std(train_data, axis=0)
        # std[std == 0] = 1e-9  # Protecție împărțire la 0
        return (train_data - mean) / std, (test_data - mean) / std

    if type == 'l1':
        train_norms = np.sum(np.abs(train_data), axis=1, keepdims=True)
        test_norms = np.sum(np.abs(test_data), axis=1, keepdims=True)
        # train_norms[train_norms == 0], test_norms[test_norms == 0] = 1.0, 1.0
        return train_data / train_norms, test_data / test_norms

    if type == 'l2':
        train_norms = np.sqrt(np.sum(train_data ** 2, axis=1, keepdims=True))
        test_norms = np.sqrt(np.sum(test_data ** 2, axis=1, keepdims=True))
        # train_norms[train_norms == 0], test_norms[test_norms == 0] = 1.0, 1.0
        return train_data / train_norms, test_data / test_norms

    raise ValueError("Tip de normalizare necunoscut! Alege: None, 'standard', 'l1', 'l2'")

class BagOfWords:
    def __init__(self):
        self.vocabulary = {}
        self.words_list = []

    def build_vocabulary(self, data):
        current_id = 0
        for message in data:
            for word in message:
                if word not in self.vocabulary:
                    self.vocabulary[word] = current_id
                    self.words_list.append(word)
                    current_id += 1

    def get_features(self, data):
        features_matrix = np.zeros((len(data), len(self.vocabulary)), dtype=np.float32)
        for sample_idx, message in enumerate(data):
            for word in message:
                if word in self.vocabulary:
                    word_idx = self.vocabulary[word]
                    features_matrix[sample_idx, word_idx] += 1
        return features_matrix


bow = BagOfWords()
bow.build_vocabulary(train_sentences)

X_train_counts = bow.get_features(train_sentences)
X_test_counts = bow.get_features(test_sentences)

X_train_scaled, X_test_scaled = normalize_data(X_train_counts, X_test_counts, type='l2')

svm_model = svm.SVC(kernel='linear', C=1.0)
svm_model.fit(X_train_scaled, train_labels)

y_pred = svm_model.predict(X_test_scaled)
acc = accuracy_score(test_labels, y_pred)
#f1-score e practic media armonica intre precision si recall (cate din cele clasificate ca spam chiar erau spam) (cate spam uri au fost prinse bine)
f1 = f1_score(test_labels, y_pred, average='binary')

print(f"Accuracy: {acc * 100:.2f}%")
print(f"F1-Score: {f1 * 100:.2f}%\n")


weights = svm_model.coef_[0]
sorted_indices = np.argsort(weights)

negative_words = [bow.words_list[idx] for idx in sorted_indices[:10]]
positive_words = [bow.words_list[idx] for idx in sorted_indices[-10:]][::-1]

print("Cele mai spammy cuvinte:\n", negative_words)
print("\nCele mai reale cuvinte:\n", positive_words)

--- REZULTATE EVALUARE ---
Accuracy: 98.42%
F1-Score: 94.09%

--- ANALIZĂ CUVINTE CHEIE ---
The first 10 negative words (spam) are:
 ['&lt#&gt', 'me', 'i', 'Going', 'him', 'Ok', 'I', 'Ill', 'my', 'Im']

The first 10 positive words (non-spam) are:
 ['STOP', 'Txt', 'Call', '&', 'txt', 'FREE', 'CALL', 'mobile', 'To', 'Text']
